# 4. BIẾN ĐỔI VÀ LỰA CHỌN ĐẶC TRƯNG (FEATURE ENGINEERING)
Chuyển đổi dữ liệu thô thành các định dạng mà thuật toán máy học có thể tiếp nhận hiệu
quả.

1. Xác định biến mục tiêu
2. Chuẩn hóa dữ liệu (Scaling)
3. Mã hóa dữ liệu danh mục

# [*] Tải thư viện và tập dữ liệu

In [1]:
from sklearn.preprocessing import StandardScaler
import pandas as pd
import warnings
import re
import seaborn as sns
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
print("Các thư viện đã sẵn sàng!")


# Đọc dữ liệu từ file CSV
df_fe = pd.read_csv('../data/processed/spotify_cleaned_scaled_v3.csv')
print(f"Đã tải dữ liệu: {df_fe.shape[0]} dòng, {df_fe.shape[1]} cột")

Các thư viện đã sẵn sàng!
Đã tải dữ liệu: 12476 dòng, 25 cột


## 1. Xác định biến mục tiêu: 
Tạo nhãn nhị phân is_hit dựa trên ngưỡng Popularity >= 70 phục vụ cho bài toán Classification.


In [2]:
# Xác định biến mục tiêu là Popularity
pop_col = 'track_popularity'
if pop_col in df_fe.columns:
    df_fe['is_hit'] = (df_fe[pop_col] >= 70).astype(int)
    print(f"Đã tạo nhãn 'is_hit' vỡi ngưỡng lỡn hơn bằng 70.")
    print(df_fe['is_hit'].value_counts())
    print("-"*40)
else: 
    print(f"Không tìm thấy cột '{pop_col}' trong dữ liệu. Vui lòng kiểm tra lại.")


Đã tạo nhãn 'is_hit' vỡi ngưỡng lỡn hơn bằng 70.
is_hit
0    8302
1    4174
Name: count, dtype: int64
----------------------------------------


## 2. Chuẩn hóa dữ liệu với StandardScaler
Áp dụng StandardScaler để đưa các biến có thang đo khác nhau (như tempo và valence) về cùng một phân phối chuẩn, đảm bảo tính công bằng cho các thuật toán dựa trên khoảng cách như K-means.

In [3]:
audio_features = ['danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness',
                  'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
ft_scaler = [f for f in audio_features if f in df_fe.columns]
scaler = StandardScaler()
df_fe[ft_scaler] = scaler.fit_transform(df_fe[ft_scaler])
print("Đã chuẩn hóa các đặc trưng âm thanh.")
print("Mẫu 5 dòng đầu tiên sau khi scale:")
print(df_fe[ft_scaler].head())
print("-"*40)



Đã chuẩn hóa các đặc trưng âm thanh.
Mẫu 5 dòng đầu tiên sau khi scale:
   danceability    energy       key  loudness      mode  speechiness  \
0       1.10236 -0.355309  1.962431  0.210754  1.209016    -0.479132   
1       1.10236 -0.355309  1.962431  0.210754  1.209016    -0.479132   
2       0.38711 -1.756868 -0.347936 -0.561848 -0.827119    -0.886576   
3      -0.40984  0.454242 -0.677989  0.267307 -0.827119    -0.408499   
4       1.10236 -0.355309  1.962431  0.210754  1.209016    -0.479132   

   acousticness  instrumentalness  liveness   valence     tempo  
0      0.109313         -0.640617 -0.984167 -0.766012 -0.404196  
1      0.109313         -0.640617 -0.984167 -0.766012 -0.404196  
2      0.694888         -0.631198 -1.066968 -0.364616 -1.991046  
3     -0.599521          1.849726 -0.400514  0.292697  0.238990  
4      0.109313         -0.640617 -0.984167 -0.766012 -0.404196  
----------------------------------------


## 3. Mã hóa dữ liệu danh mục
Sử dụng kỹ thuật One-hot Encoding cho cột artist_genres để phục vụ cho thuật toán Apriori và ANN.

Xử lý cột artist_geners, tách các list thuộc tính thành từng thuộc tính riêng biệt, trách chồng lấp tạo ra nhiều cột không cần thiết.

In [4]:
# Lấy danh sách các thể loại âm nhạc
genre_col = 'artist_genres'

# Xóa ký tự đặc biệt ([], '', "") và tách riêng các thể loại 
if genre_col in df_fe.columns:
    def parse_genres_aggressive(val):
        if pd.isna(val):
            return ""
        val = str(val)
        val = re.sub(r'[\[\]\'"]', '', val)
        genres = [g.strip() for g in val.split(',') if g.strip()]
        return "|".join(genres)

    df_fe['genres_cleaned'] = df_fe[genre_col].apply(parse_genres_aggressive)

    # Tách các thể loại thành cột riêng biệt
    print(f"Đang bóc tách và tạo cột cho từng thể loại riêng biệt...")
    genre_dummies = df_fe['genres_cleaned'].str.get_dummies(sep='|')
    
    genre_dummies = genre_dummies.add_prefix('genre_')

    df_fe = pd.concat([df_fe, genre_dummies], axis=1)
    df_fe = df_fe.drop(columns=[genre_col, 'genres_cleaned'])
    
    print(f"Mã hóa nhãn thành công")
    print(f"Tổng số cột thể loại được tạo ra: {genre_dummies.shape[1]} cột")
else:
    print(f"Không tìm thấy cột '{genre_col}'.")

# Xem thử tên các cột để kiểm chứng
print("Xem thử tên các cột mới được tạo:")
genre_cols_only = [col for col in df_fe.columns if col.startswith('genre_')]
print(genre_cols_only) 

# Lưu file ML mới
final_ml_path = '../data/processed/spotify_for_ml.csv'
df_fe.to_csv(final_ml_path, index=False)
print(f"Dữ liệu SẠCH 100% đã được lưu vào: {final_ml_path}")

Đang bóc tách và tạo cột cho từng thể loại riêng biệt...
Mã hóa nhãn thành công
Tổng số cột thể loại được tạo ra: 301 cột
Xem thử tên các cột mới được tạo:
['genre_acid house', 'genre_acid rock', 'genre_acoustic country', 'genre_acoustic pop', 'genre_afro r&b', 'genre_afrobeat', 'genre_afrobeats', 'genre_afropiano', 'genre_afropop', 'genre_afroswing', 'genre_alt country', 'genre_alternative dance', 'genre_alternative metal', 'genre_alternative pop', 'genre_alternative r&b', 'genre_alternative rock', 'genre_ambient', 'genre_ambient folk', 'genre_americana', 'genre_anime', 'genre_anti-folk', 'genre_aor', 'genre_arena rock', 'genre_argentine trap', 'genre_art pop', 'genre_art rock', 'genre_azonto', 'genre_bachata', 'genre_ballroom vogue', 'genre_baroque pop', 'genre_bass music', 'genre_bedroom pop', 'genre_big room', 'genre_bluegrass', 'genre_blues', 'genre_blues rock', 'genre_boom bap', 'genre_brazilian bass', 'genre_brazilian funk', 'genre_brazilian pop', 'genre_breakcore', 'genre_britp

In [5]:

# ===== VERIFICATION: Check ML-Ready File =====
print("="*100)
print("✅ KIỂM TRA FILE ML-READY: spotify_for_ml.csv")
print("="*100)

import os
ml_file_path = "../data/processed/spotify_for_ml.csv"

if os.path.exists(ml_file_path):
    df_ml = pd.read_csv(ml_file_path)
    
    print(f"\n📊 KÍCH THƯỚC:")
    print(f"   Rows: {len(df_ml):,}")
    print(f"   Columns: {len(df_ml.columns):,}")
    print(f"   Shape: {df_ml.shape}")
    print(f"   File size: {os.path.getsize(ml_file_path) / (1024*1024):.2f} MB")
    
    print(f"\n💾 MEMORY USAGE:")
    print(f"   Total: {df_ml.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    if 'is_hit' in df_ml.columns:
        print(f"\n🎯 TARGET VARIABLE (is_hit):")
        print(f"   Distribution:")
        print(df_ml['is_hit'].value_counts())
        print(f"   Hit ratio: {df_ml['is_hit'].sum() / len(df_ml) * 100:.2f}%")
        print(f"   Class balance: {'✅ Balanced' if (df_ml['is_hit'].sum() / len(df_ml)) >= 0.3 else '⚠️ Imbalanced'}")
    
    audio_cols = ['danceability', 'energy', 'key', 'loudness', 'mode', 
                  'speechiness', 'acousticness', 'instrumentalness', 
                  'liveness', 'valence', 'tempo']
    audio_in_ml = [c for c in audio_cols if c in df_ml.columns]
    print(f"\n🎵 AUDIO FEATURES:")
    print(f"   Available: {len(audio_in_ml)}/{len(audio_cols)}")
    
    genre_cols = [c for c in df_ml.columns if c.startswith('genre_')]
    print(f"\n🎸 GENRE FEATURES:")
    print(f"   Total: {len(genre_cols)}")
    
    nan_count = df_ml.isna().sum().sum()
    print(f"\n⚠️ MISSING VALUES:")
    print(f"   Total NaN: {nan_count}")
    if nan_count == 0:
        print(f"   ✅ NO MISSING VALUES - READY FOR ML!")
    
    print(f"\n🔧 DATA TYPES:")
    print(df_ml.dtypes.value_counts())
    
    print(f"\n📝 COLUMN LIST:")
    print(f"   {list(df_ml.columns)}")
    
    print("\n" + "="*100)
    print("✅ FILE READY FOR MACHINE LEARNING MODELS!")
    print("="*100)
else:
    print(f"❌ File not found: {ml_file_path}")


✅ KIỂM TRA FILE ML-READY: spotify_for_ml.csv

📊 KÍCH THƯỚC:
   Rows: 12,476
   Columns: 326
   Shape: (12476, 326)
   File size: 11.50 MB

💾 MEMORY USAGE:
   Total: 35.67 MB

🎯 TARGET VARIABLE (is_hit):
   Distribution:
is_hit
0    8302
1    4174
Name: count, dtype: int64
   Hit ratio: 33.46%
   Class balance: ✅ Balanced

🎵 AUDIO FEATURES:
   Available: 11/11

🎸 GENRE FEATURES:
   Total: 301

⚠️ MISSING VALUES:
   Total NaN: 4

🔧 DATA TYPES:
int64      305
float64     13
object       7
bool         1
Name: count, dtype: int64

📝 COLUMN LIST:
   ['track_id', 'track_name', 'track_number', 'track_popularity', 'explicit', 'artist_name', 'artist_popularity', 'artist_followers', 'album_id', 'album_name', 'album_release_date', 'album_total_tracks', 'album_type', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'is_hit', 'genre_acid house', 'genre_acid rock', 'genre_acoustic country', 'genre_acoustic pop', '